# Segmentación de clientes mediante K-Means

**Asignatura:** Inteligencia Artificial  
**Estudiante:** Mejía Jefferson  
**Técnica:** aprendizaje no supervisado

## 1. Introducción

La segmentación de clientes permite identificar grupos con características y hábitos de consumo similares. En este cuaderno se presenta un ejercicio académico de aprendizaje no supervisado usando K-Means para explorar esos grupos sin disponer de etiquetas previas.

## 2. Planteamiento del problema

Una empresa desea conocer los perfiles de sus clientes para adaptar campañas, promociones y estrategias de fidelización. Sin embargo, no dispone de una etiqueta previa que indique a qué segmento pertenece cada cliente. Por ello, se requiere descubrir grupos naturales a partir de sus características demográficas y de consumo.

## 3. Objetivos

**Objetivo general:** Aplicar **K-Means** para agrupar clientes con características similares de edad, ingreso mensual y gasto mensual, e interpretar los segmentos obtenidos para apoyar decisiones comerciales.

## 4. Marco teórico

K-Means es un algoritmo de aprendizaje no supervisado que asigna cada observación al centroide más cercano y busca minimizar la suma de cuadrados dentro de los grupos (inercia). No utiliza una variable objetivo ni etiquetas conocidas.

El método fue propuesto por Stuart Lloyd (1957; publicado en 1982) y popularizado por James MacQueen (1967). Debido a que K-Means depende de distancias, las variables se estandarizan antes de ajustar el modelo.


## 5. Descripción de los datos

El conjunto de datos contiene 120 observaciones de clientes. Para este ejercicio se emplean variables demográficas y de consumo que permiten explorar la segmentación.


### 5.1. Variables analizadas

- **Edad:** años del cliente.
- **Ingreso mensual:** ingreso estimado en dólares.
- **Gasto mensual:** gasto estimado en dólares.


### 5.2. Procedencia y alcance de los datos

Los datos fueron generados de forma sintética para practicar la metodología; no corresponden a clientes reales. Por tanto, los resultados ilustran el proceso de segmentación, pero no deben convertirse directamente en decisiones comerciales. Con datos reales se debería revisar valores faltantes, duplicados y valores atípicos antes del modelado.

## 6. Metodología

Las variables tienen escalas diferentes (años y dólares). Se utiliza `StandardScaler` para que ninguna variable domine el cálculo de las distancias solo por su magnitud.

### 6.1. Importación de librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Identidad visual centralizada para las figuras del informe.
COLOR_PRINCIPAL = "#2563EB"
COLOR_DESTACADO = "#F97316"
COLOR_OSCURO = "#0F172A"
PALETA_CLUSTERS = ["#4E79A7", "#59A14F", "#E15759", "#B07AA1", "#F28E2B", "#76B7B2"]

plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.facecolor": "#F8FAFC",
    "axes.edgecolor": "#CBD5E1",
    "axes.labelcolor": "#334155",
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size": 10,
    "xtick.color": "#475569",
    "ytick.color": "#475569",
    "legend.frameon": True,
    "legend.facecolor": "white",
    "legend.edgecolor": "#CBD5E1",
})


### 6.2. Generación del conjunto de datos

In [ ]:
# Perfiles sintéticos: jóvenes de bajo gasto, clientes regulares,
# clientes de alto valor y clientes maduros con gasto moderado.
perfiles = [
    (24, 1700, 300),
    (35, 3600, 1100),
    (43, 7200, 2700),
    (58, 5000, 1500),
]

datos = []
for edad_media, ingreso_medio, gasto_medio in perfiles:
    datos.append(np.column_stack([
        np.random.normal(edad_media, 4, 30),
        np.random.normal(ingreso_medio, 700, 30),
        np.random.normal(gasto_medio, 250, 30),
    ]))

clientes = pd.DataFrame(
    np.vstack(datos), columns=["Edad", "Ingreso mensual", "Gasto mensual"]
)
clientes["Edad"] = clientes["Edad"].round().clip(18, 70).astype(int)
clientes[["Ingreso mensual", "Gasto mensual"]] = clientes[
    ["Ingreso mensual", "Gasto mensual"]
].clip(lower=0).round(2)

clientes.head()

### 6.3. Preparación de los datos


In [ ]:
variables = ["Edad", "Ingreso mensual", "Gasto mensual"]
X = clientes[variables]
escalador = StandardScaler()
X_escalado = escalador.fit_transform(X)

### 6.4. Selección del número de grupos: método del codo

El método del codo compara la inercia para distintos valores de **k**. Se selecciona el punto donde la reducción de la inercia comienza a disminuir de forma notable; ese cambio de pendiente sugiere un número de grupos adecuado.


In [ ]:
resultados_k = []
for k in range(2, 7):
    modelo = KMeans(n_clusters=k, n_init=20, random_state=42)
    modelo.fit(X_escalado)
    resultados_k.append({"k": k, "Inercia": modelo.inertia_})

evaluacion = pd.DataFrame(resultados_k)
evaluacion


In [ ]:
# Se identifica el punto más alejado de la recta entre los extremos de la curva.
# Es una aproximación automática y reproducible del "codo".
k_valores = evaluacion["k"].to_numpy(dtype=float)
inercia = evaluacion["Inercia"].to_numpy(dtype=float)
puntos = np.column_stack((
    (k_valores - k_valores.min()) / (k_valores.max() - k_valores.min()),
    (inercia - inercia.min()) / (inercia.max() - inercia.min()),
))
inicio, fin = puntos[0], puntos[-1]
vector_linea = fin - inicio
vectores_puntos = puntos - inicio
distancias = np.abs(
    vector_linea[0] * vectores_puntos[:, 1] - vector_linea[1] * vectores_puntos[:, 0]
) / np.linalg.norm(vector_linea)
k_optimo = int(k_valores[np.argmax(distancias)])

print(f"Valor de k sugerido por el método del codo: {k_optimo}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(
    evaluacion["k"], evaluacion["Inercia"],
    marker="o", markersize=7, linewidth=2.5, color=COLOR_PRINCIPAL
)
ax.scatter(k_optimo, evaluacion.loc[evaluacion["k"] == k_optimo, "Inercia"],
           s=110, color=COLOR_DESTACADO, zorder=3, label=f"Codo sugerido: k = {k_optimo}")
ax.annotate(
    f"k = {k_optimo}",
    xy=(k_optimo, evaluacion.loc[evaluacion["k"] == k_optimo, "Inercia"].iloc[0]),
    xytext=(10, 15), textcoords="offset points", color=COLOR_DESTACADO, fontweight="bold"
)
ax.set_title("Selección del número de clusters: método del codo", loc="left", pad=12)
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Inercia")
ax.set_xticks(evaluacion["k"])
ax.grid(axis="y", alpha=0.35, linestyle="--")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.legend(loc="best")
fig.tight_layout()
plt.show()


### 6.5. Ajuste de K-Means

El modelo se ajusta usando el valor de `k` sugerido por el método del codo. El valor calculado se muestra al ejecutar la sección anterior.


In [ ]:
k = k_optimo
modelo_final = KMeans(n_clusters=k, n_init=20, random_state=42)
clientes["Cluster"] = modelo_final.fit_predict(X_escalado)

centroides = pd.DataFrame(
    escalador.inverse_transform(modelo_final.cluster_centers_), columns=variables
)
centroides.index.name = "Cluster"
centroides.round(2)

## 7. Resultados

El gráfico usa edad y gasto mensual para facilitar la lectura. El modelo, no obstante, considera las tres variables definidas.

### 7.1. Visualización de los segmentos

El gráfico usa edad y gasto mensual para facilitar la lectura. El modelo considera las tres variables definidas.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
paleta = PALETA_CLUSTERS[:k]

for color, cluster in zip(paleta, sorted(clientes["Cluster"].unique())):
    grupo = clientes[clientes["Cluster"] == cluster]
    ax.scatter(
        grupo["Edad"], grupo["Gasto mensual"], s=52, alpha=0.82,
        color=color, edgecolor="white", linewidth=0.6,
        label=f"Cluster {cluster + 1}"
    )

ax.scatter(
    centroides["Edad"], centroides["Gasto mensual"],
    c=COLOR_OSCURO, marker="X", s=230, edgecolor="white", linewidth=1.2,
    label="Centroides", zorder=4
)
ax.set_title("Segmentación de clientes mediante K-Means", loc="left", pad=12)
ax.set_xlabel("Edad (años)")
ax.set_ylabel("Gasto mensual (USD)")
ax.grid(alpha=0.28, linestyle="--")
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
ax.legend(title="Segmentos", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()


### 7.2. Perfiles de clientes

La siguiente tabla resume cada grupo usando los promedios originales, no estandarizados. Los nombres de perfil son interpretaciones de apoyo y deben validarse con conocimiento real del negocio.


In [ ]:
resumen = clientes.groupby("Cluster")[variables].mean().round(2)
resumen["Cantidad de clientes"] = clientes.groupby("Cluster").size()

# Los perfiles se ordenan por gasto promedio; su interpretación final se realiza
# después de revisar los resultados obtenidos.
orden_gasto = resumen["Gasto mensual"].sort_values().index.tolist()
nombres = {
    cluster: f"Perfil de gasto {posicion + 1}"
    for posicion, cluster in enumerate(orden_gasto)
}
resumen["Perfil propuesto"] = resumen.index.map(nombres)
resumen = resumen[["Perfil propuesto", "Cantidad de clientes", *variables]]
resumen


## 8. Conclusiones

## 9. Recomendaciones

## 10. Referencias bibliográficas

Lloyd, S. (1982). *Least squares quantization in PCM*. IEEE Transactions on Information Theory, 28(2), 129–137. https://doi.org/10.1109/TIT.1982.1056489

MacQueen, J. (1967). *Some methods for classification and analysis of multivariate observations*. Proceedings of the Fifth Berkeley Symposium on Mathematical Statistics and Probability. https://projecteuclid.org/euclid.bsmsp/1200512992
